# 07 - Ray-ID CrossFormer

The fine-tuned SOTA checkpoint reached weak Top-1 because the previous architecture only saw shuffled candidate slots and RF histories. In the ray-tracing cache, labels are remapped after candidate shuffling, but the old model never receives the physical `candidate_ids`, so it is partly blind to the cell identity transition it must learn.

This notebook trains a new architecture on the same ray-tracing cache:

- temporal RF encoder per candidate slot;
- physical cell ID embedding;
- serving-cell indicator embedding;
- masked self-attention across candidate cells;
- horizon-conditioned cross-attention decoder;
- direct `(horizon, candidate)` scoring.

The comparison target is `metrics/ray_tracing_finetune/ray_finetune_metadata.json`.

In [19]:
# Section 1 - Environment and paths
import os, sys, json, pickle, logging, datetime, math, warnings
from pathlib import Path
from typing import Dict, Optional

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix, top_k_accuracy_score
from sklearn.utils.class_weight import compute_class_weight

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, mixed_precision
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("TensorFlow is required. Run this notebook in the project ML environment.") from exc

sns.set_theme(style="whitegrid", font_scale=1.0)


def find_project_root(start: Optional[Path] = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "dataset" / "ray_tracing_finetune_cache" / "train.npz").exists():
            return p
    return Path("../../").resolve()


ROOT = find_project_root()
PATHS = {
    "cache": ROOT / "dataset" / "ray_tracing_finetune_cache",
    "prev_metrics": ROOT / "metrics" / "ray_tracing_finetune" / "ray_finetune_metadata.json",
    "metrics": ROOT / "metrics" / "ray_id_crossformer",
    "models": ROOT / "models",
    "tb_logs": ROOT / "tb_logs" / "ray_id_crossformer",
}
for p in [PATHS["metrics"], PATHS["models"], PATHS["tb_logs"]]:
    p.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(PATHS["metrics"] / "ray_id_crossformer.log", mode="w"),
    ],
)
log = logging.getLogger("ray_id_crossformer")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
if gpus:
    mixed_precision.set_global_policy(mixed_precision.Policy("mixed_float16"))
else:
    mixed_precision.set_global_policy(mixed_precision.Policy("float32"))
log.info("Root=%s GPUs=%d", ROOT, len(gpus))

RUN_TRAIN = True

10:09:48 | INFO     | Root=/home/wassimmchichi/Downloads/Handover_projects GPUs=1


In [20]:
# Section 2 - Inspect previous results and cache geometry
previous = json.load(open(PATHS["prev_metrics"])) if PATHS["prev_metrics"].exists() else {}
print("Previous source model:", previous.get("source_model"))
print("Previous baseline:", previous.get("baseline_metrics"))
print("Previous fine-tuned:", previous.get("finetuned_metrics"))

cache_meta = json.load(open(PATHS["cache"] / "meta.json"))
print("Cache labeling:", cache_meta.get("labeling"))
print("UE split:", cache_meta.get("ue_split"))

data = {s: dict(np.load(PATHS["cache"] / f"{s}.npz")) for s in ["train", "val", "test"]}
for split, d in data.items():
    print(split, {k: v.shape for k, v in d.items()})
    vals, cnts = np.unique(d["y"], return_counts=True)
    print(" labels", dict(zip(vals.tolist(), cnts.tolist())))

C = int(data["train"]["X"].shape[1])
T = int(data["train"]["X"].shape[2])
F = int(data["train"]["X"].shape[3])
H = int(data["train"]["y"].shape[1])
ALL_LABELS = list(range(C))
log.info("Geometry C=%d T=%d F=%d H=%d", C, T, F, H)

Previous source model: mtl_transformer
Previous baseline: {'baseline_top1': 0.21404456448345713, 'baseline_top3': 0.638487508440243, 'baseline_top5': 0.9832545577312627, 'baseline_rsrp_mae_dbm': 8.54693603515625}
Previous fine-tuned: {'finetuned_top1': 0.2811613774476705, 'finetuned_top3': 0.7254557731262661, 'finetuned_top5': 0.9856853477380149, 'finetuned_rsrp_mae_dbm': 8.10690689086914}
Cache labeling: candidate axis is horizon-target union plus anchor neighbours, shuffled per window; y remapped from optimal_cell_id
UE split: {'train': ['MA_UE_0004', 'MA_UE_0002', 'MA_UE_0001'], 'val': ['MA_UE_0003'], 'test': ['MA_UE_0005']}
train {'X': (4443, 10, 25, 4), 'M': (4443, 10), 'y': (4443, 5), 'r': (4443, 5), 'candidate_ids': (4443, 10), 'serving_id': (4443,)}
 labels {0: 2230, 1: 2234, 2: 2183, 3: 2290, 4: 2253, 5: 2206, 6: 2231, 7: 2221, 8: 2182, 9: 2185}
val {'X': (1481, 10, 25, 4), 'M': (1481, 10), 'y': (1481, 5), 'r': (1481, 5), 'candidate_ids': (1481, 10), 'serving_id': (1481,)}
 la

In [21]:
# Section 3 - Candidate ID vocabulary, serving flags, and tf.data
# Use train IDs for the supervised vocabulary, with OOV for unseen ray cells.
train_ids = sorted(set(map(int, data["train"]["candidate_ids"].ravel())) - {0})
ID_TO_INDEX = {cid: i + 1 for i, cid in enumerate(train_ids)}
OOV_INDEX = 0
VOCAB_SIZE = len(ID_TO_INDEX) + 1
print("Train candidate ID vocab size:", VOCAB_SIZE)


def encode_candidate_ids(arr: np.ndarray) -> np.ndarray:
    out = np.zeros_like(arr, dtype=np.int32)
    for cid, idx in ID_TO_INDEX.items():
        out[arr == cid] = idx
    return out


def serving_flags(candidate_ids: np.ndarray, serving_id: np.ndarray) -> np.ndarray:
    return (candidate_ids == serving_id[:, None]).astype(np.float32)


def one_hot_y(y: np.ndarray) -> np.ndarray:
    return tf.one_hot(y.astype(np.int32), depth=C).numpy().astype(np.float32)

present = np.unique(data["train"]["y"].ravel())
class_weight_values = compute_class_weight("balanced", classes=present, y=data["train"]["y"].ravel())
CLASS_WEIGHT = {int(c): float(w) for c, w in zip(present, class_weight_values)}
print("Class weights:", {k: round(v, 3) for k, v in CLASS_WEIGHT.items()})


def y_weights(y: np.ndarray) -> np.ndarray:
    return np.vectorize(lambda v: CLASS_WEIGHT.get(int(v), 1.0))(y).astype(np.float32)


def make_arrays(split: str):
    d = data[split]
    inputs = {
        "cells": d["X"].astype(np.float32),
        "mask": d["M"].astype(np.float32),
        "candidate_ids": encode_candidate_ids(d["candidate_ids"]).astype(np.int32),
        "serving_flag": serving_flags(d["candidate_ids"], d["serving_id"]).astype(np.float32),
        "horizon_ids": np.tile(np.arange(H, dtype=np.int32), (len(d["y"]), 1)),
    }
    targets = one_hot_y(d["y"])
    weights = y_weights(d["y"])
    return inputs, targets, weights

arrays = {split: make_arrays(split) for split in ["train", "val", "test"]}


def make_ds(split: str, shuffle: bool = False) -> tf.data.Dataset:
    inputs, targets, weights = arrays[split]
    with tf.device("/CPU:0"):
        ds = tf.data.Dataset.from_tensor_slices((inputs, targets, weights))
    if shuffle:
        ds = ds.shuffle(len(targets), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(HP["BATCH_SIZE"], drop_remainder=False).prefetch(tf.data.AUTOTUNE)

Train candidate ID vocab size: 172
Class weights: {0: 0.996, 1: 0.994, 2: 1.018, 3: 0.97, 4: 0.986, 5: 1.007, 6: 0.996, 7: 1.0, 8: 1.018, 9: 1.017}


In [22]:
# Section 4 - Hyperparameters
HP = {
    "BATCH_SIZE": 64,
    "EPOCHS": 90,
    "LR": 3e-4,
    "MIN_LR": 2e-6,
    "PATIENCE": 14,
    "D_MODEL": 128,
    "ID_DIM": 32,
    "GRU_UNITS": 96,
    "N_HEADS": 4,
    "FF_DIM": 256,
    "N_BLOCKS": 3,
    "DROPOUT": 0.18,
    "FOCAL_GAMMA": 1.5,
    "FOCAL_ALPHA": 0.75,
    "LABEL_SMOOTHING": 0.01,
}

ds_tr = make_ds("train", shuffle=True)
ds_va = make_ds("val")
ds_te = make_ds("test")
log.info("Batches train=%d val=%d test=%d", len(ds_tr), len(ds_va), len(ds_te))

10:09:48 | INFO     | Batches train=70 val=24 test=24


In [23]:
# Section 5 - Loss, metrics, and custom attention block
class MaskedSelfAttentionBlock(keras.layers.Layer):
    def __init__(self, d_model, n_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.cfg = dict(d_model=d_model, n_heads=n_heads, ff_dim=ff_dim, dropout=dropout)
        key_dim = max(8, d_model // n_heads)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6, dtype="float32")
        self.norm2 = layers.LayerNormalization(epsilon=1e-6, dtype="float32")
        self.attn = layers.MultiHeadAttention(num_heads=n_heads, key_dim=key_dim, dropout=dropout, dtype="float32")
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="gelu"),
            layers.Dropout(dropout),
            layers.Dense(d_model),
        ])
        self.drop = layers.Dropout(dropout)

    def call(self, x, mask, training=False):
        attn_mask = tf.cast(mask[:, tf.newaxis, :], tf.bool)
        a = self.attn(self.norm1(x), self.norm1(x), attention_mask=attn_mask, training=training)
        x = x + self.drop(a, training=training)
        x = x + self.drop(self.ffn(self.norm2(x), training=training), training=training)
        return x * tf.cast(mask[:, :, tf.newaxis], x.dtype)

    def get_config(self):
        return {**super().get_config(), **self.cfg}


def focal_categorical_loss(gamma=1.5, alpha=0.75, label_smoothing=0.01):
    def _loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1.0 - 1e-7)
        if label_smoothing > 0:
            y_true = y_true * (1.0 - label_smoothing) + label_smoothing / tf.cast(tf.shape(y_true)[-1], tf.float32)
        p_t = tf.reduce_sum(y_true * y_pred, axis=-1)
        ce = -tf.reduce_sum(y_true * tf.math.log(y_pred), axis=-1)
        return alpha * tf.pow(1.0 - p_t, gamma) * ce
    _loss.__name__ = f"focal_cat_g{gamma}_a{alpha}"
    return _loss


def topk_metric(k: int):
    return keras.metrics.TopKCategoricalAccuracy(k=k, name=f"top{k}_acc")

LOSS_FN = focal_categorical_loss(HP["FOCAL_GAMMA"], HP["FOCAL_ALPHA"], HP["LABEL_SMOOTHING"])

In [24]:
# Section 6 - Ray-ID CrossFormer architecture

def build_ray_id_crossformer() -> keras.Model:
    d = HP["D_MODEL"]
    inp_cells = keras.Input((C, T, F), name="cells", dtype="float32")
    inp_mask  = keras.Input((C,),       name="mask",          dtype="float32")
    inp_ids   = keras.Input((C,),       name="candidate_ids", dtype="int32")
    inp_serv  = keras.Input((C,),       name="serving_flag",  dtype="float32")
    inp_h     = keras.Input((H,),       name="horizon_ids",   dtype="int32")

    # Temporal RF grammar per candidate slot.
    z = layers.Reshape((C * T, F), name="flat_cells_time")(inp_cells)
    z = layers.LayerNormalization(epsilon=1e-6, dtype="float32", name="rf_norm")(z)
    z = layers.Reshape((C, T, F), name="rf_unflatten")(z)
    temporal = layers.TimeDistributed(
        keras.Sequential([
            layers.Conv1D(64, kernel_size=3, padding="same", activation="gelu"),
            layers.Dropout(HP["DROPOUT"]),
            layers.Bidirectional(layers.GRU(HP["GRU_UNITS"], return_sequences=False)),
        ]),
        name="temporal_encoder",
    )(z)
    temporal = layers.Dense(d, activation="gelu", name="temporal_proj")(temporal)

    # Physical identity and serving context.
    id_emb  = layers.Embedding(VOCAB_SIZE, HP["ID_DIM"], mask_zero=False, name="cell_id_embedding")(inp_ids)
    id_emb  = layers.Dense(d, activation="gelu", name="cell_id_proj")(id_emb)
    serving = layers.Reshape((C, 1), name="serving_flag_exp")(inp_serv)
    serving = layers.Dense(d, activation="gelu", name="serving_proj")(serving)

    x = layers.Concatenate(name="token_concat")([temporal, id_emb, serving])
    x = layers.Dense(d, activation="gelu", name="token_proj")(x)
    x = layers.Dropout(HP["DROPOUT"], name="token_drop")(x)
    mask_exp = layers.Reshape((C, 1), name="token_mask_exp")(inp_mask)
    x = layers.Multiply(name="token_masked")([x, mask_exp])

    for i in range(HP["N_BLOCKS"]):
        x = MaskedSelfAttentionBlock(d, HP["N_HEADS"], HP["FF_DIM"], HP["DROPOUT"], name=f"cell_block_{i}")(x, inp_mask)

    # Cast backbone output to float32 before all scoring layers to guard
    # against mixed-precision policies that leave x in float16.
    x = layers.Lambda(lambda t: tf.cast(t, tf.float32), name="backbone_fp32_cast")(x)

    # Horizon queries ask different questions of the same candidate set.
    hq = layers.Embedding(H, d, dtype="float32", name="horizon_embedding")(inp_h)
    cross_mask = layers.Lambda(
        lambda m: tf.tile(tf.cast(m[:, tf.newaxis, :], tf.bool), [1, H, 1]),
        name="horizon_cross_mask",
    )(inp_mask)
    hctx = layers.MultiHeadAttention(
        num_heads=HP["N_HEADS"], key_dim=max(8, d // HP["N_HEADS"]),
        dropout=HP["DROPOUT"], dtype="float32", name="horizon_cross_attention",
    )(hq, x, attention_mask=cross_mask)
    hq   = layers.Lambda(lambda t: tf.cast(t, tf.float32), name="horizon_query_fp32")(hq)
    hctx = layers.Add(name="horizon_residual", dtype="float32")([hctx, hq])
    hctx = layers.LayerNormalization(epsilon=1e-6, dtype="float32", name="horizon_ctx_norm")(hctx)

    # Bilinear scoring: explicitly cast both projections to float32 before
    # the einsum so the Lambda never receives mixed-dtype tensors.
    cand_proj = layers.Dense(d, use_bias=False, dtype="float32", name="candidate_score_proj")(x)
    hor_proj  = layers.Dense(d, use_bias=False, dtype="float32", name="horizon_score_proj")(hctx)
    cand_proj = layers.Lambda(lambda t: tf.cast(t, tf.float32), name="cand_proj_fp32")(cand_proj)
    hor_proj  = layers.Lambda(lambda t: tf.cast(t, tf.float32), name="hor_proj_fp32")(hor_proj)
    dot_logits = layers.Lambda(
        lambda zs: tf.einsum("bhd,bcd->bhc", zs[0], zs[1]) / tf.sqrt(tf.cast(d, tf.float32)),
        name="bilinear_horizon_candidate_logits",
        dtype="float32",
    )([hor_proj, cand_proj])

    slot_bias = layers.TimeDistributed(layers.Dense(H, dtype="float32"), name="candidate_horizon_bias")(x)
    slot_bias = layers.Permute((2, 1), name="candidate_horizon_bias_permuted")(slot_bias)
    slot_bias = layers.Lambda(lambda t: tf.cast(t, tf.float32), name="slot_bias_fp32")(slot_bias)
    logits    = layers.Add(name="logits", dtype="float32")([dot_logits, slot_bias])
    mask_bias = layers.Lambda(
        lambda m: tf.reshape(tf.cast((1.0 - m), tf.float32) * (-1e9), (-1, 1, C)),
        name="mask_bias",
        dtype="float32",
    )(inp_mask)
    probs = layers.Softmax(axis=-1, dtype="float32", name="cls_output")(
        layers.Add(name="masked_logits", dtype="float32")([logits, mask_bias])
    )

    return keras.Model(
        inputs={"cells": inp_cells, "mask": inp_mask, "candidate_ids": inp_ids, "serving_flag": inp_serv, "horizon_ids": inp_h},
        outputs=probs,
        name="Ray_ID_CrossFormer",
    )

model = build_ray_id_crossformer()
model.summary(line_length=110, expand_nested=False)


Model: "Ray_ID_CrossFormer"
______________________________________________________________________________________________________________
 Layer (type)                    Output Shape                     Param #    Connected to                     
 cells (InputLayer)              [(None, 10, 25, 4)]              0          []                               
                                                                                                              
 flat_cells_time (Reshape)       (None, 250, 4)                   0          ['cells[0][0]']                  
                                                                                                              
 rf_norm (LayerNormalization)    (None, 250, 4)                   8          ['flat_cells_time[0][0]']        
                                                                                                              
 rf_unflatten (Reshape)          (None, 10, 25, 4)                0          ['rf_no

In [25]:
# Section 7 - Compile and train
RUN_TRAIN = True   # set to False to skip training and load from checkpoint

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=HP["LR"], weight_decay=1e-4, clipnorm=1.0),
    loss=LOSS_FN,
    metrics=[keras.metrics.CategoricalAccuracy(name="top1_acc"), topk_metric(3), topk_metric(5)],
)

CKPT_PATH  = PATHS["models"] / "best_ray_id_crossformer.keras"
FINAL_PATH = PATHS["models"] / "ray_id_crossformer_final.keras"

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_top1_acc", mode="max", patience=HP["PATIENCE"],
        min_delta=1e-4, restore_best_weights=True, verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        str(CKPT_PATH), monitor="val_top1_acc", mode="max", save_best_only=True, verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_top1_acc", mode="max", factor=0.5, patience=5,
        min_lr=HP["MIN_LR"], verbose=1,
    ),
    keras.callbacks.CSVLogger(str(PATHS["metrics"] / "history.csv"), append=False),
    keras.callbacks.TensorBoard(log_dir=str(PATHS["tb_logs"]), update_freq="epoch"),
]

if RUN_TRAIN:
    history = model.fit(ds_tr, validation_data=ds_va, epochs=HP["EPOCHS"], callbacks=callbacks, verbose=1)
else:
    history = None
    print("Skipping training — will load checkpoint in Section 8.")


Epoch 1/90


I0000 00:00:1780218618.749098   39766 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


70/70 [==============================] - ETA: 0s - loss: 1.2988 - top1_acc: 0.1828 - top3_acc: 0.5277 - top5_acc: 0.8136WARNING:tensorflow:`evaluate()` received a value for `sample_weight`, but `weighted_metrics` were not provided.  Did you mean to pass metrics to `weighted_metrics` in `compile()`?  If this is intentional you can pass `weighted_metrics=[]` to `compile()` in order to silence this warning.
10:10:37 | WARNING  | `evaluate()` received a value for `sample_weight`, but `weighted_metrics` were not provided.  Did you mean to pass metrics to `weighted_metrics` in `compile()`?  If this is intentional you can pass `weighted_metrics=[]` to `compile()` in order to silence this warning.

Epoch 1: val_top1_acc improved from -inf to 0.22822, saving model to /home/wassimmchichi/Downloads/Handover_projects/models/best_ray_id_crossformer.keras
70/70 [==============================] - 49s 177ms/step - loss: 1.2988 - top1_acc: 0.1828 - top3_acc: 0.5277 - top5_acc: 0.8136 - val_loss: 0.9428

In [26]:
# Section 8 - Evaluation helpers

# Guard: re-derive paths if Cell 7 (compile/train) was skipped or errored.
if "CKPT_PATH" not in dir():
    CKPT_PATH  = PATHS["models"] / "best_ray_id_crossformer.keras"
if "FINAL_PATH" not in dir():
    FINAL_PATH = PATHS["models"] / "ray_id_crossformer_final.keras"

CUSTOM_OBJECTS = {
    "MaskedSelfAttentionBlock": MaskedSelfAttentionBlock,
    LOSS_FN.__name__: LOSS_FN,
}
if CKPT_PATH.exists():
    model = keras.models.load_model(CKPT_PATH, custom_objects=CUSTOM_OBJECTS, compile=False, safe_mode=False)
    print(f"Loaded checkpoint from {CKPT_PATH}")
else:
    print(f"No checkpoint found at {CKPT_PATH} — using in-memory model.")


def predict_split(split: str) -> np.ndarray:
    ds = make_ds(split, shuffle=False).map(lambda x, y, w: x)
    return model.predict(ds, verbose=1)


def evaluate_split(split: str, probs: np.ndarray) -> Dict[str, float]:
    y = data[split]["y"]
    y_pred = probs.argmax(axis=-1)
    y_flat = y.ravel()
    pred_flat = y_pred.ravel()
    p_flat = probs.reshape(-1, probs.shape[-1])
    metrics = {
        f"{split}_top1": float((pred_flat == y_flat).mean()),
        f"{split}_top3": float(top_k_accuracy_score(y_flat, p_flat, k=3, labels=ALL_LABELS)),
        f"{split}_top5": float(top_k_accuracy_score(y_flat, p_flat, k=5, labels=ALL_LABELS)),
    }
    print(split, metrics)
    print(classification_report(y_flat, pred_flat, labels=ALL_LABELS, target_names=[f"C{i}" for i in ALL_LABELS], digits=4, zero_division=0))
    return metrics

val_probs   = predict_split("val")
test_probs  = predict_split("test")
val_metrics  = evaluate_split("val",  val_probs)
test_metrics = evaluate_split("test", test_probs)
prev_ft = previous.get("finetuned_metrics", {})
print("Previous fine-tuned Top-1:", prev_ft.get("finetuned_top1"))
print("Ray-ID CrossFormer Top-1:", test_metrics["test_top1"])


Loaded checkpoint from /home/wassimmchichi/Downloads/Handover_projects/models/best_ray_id_crossformer.keras
24/24 [==============================] - 1s 24ms/step
val {'val_top1': 0.599594868332208, 'val_top3': 0.9085752869682647, 'val_top5': 0.987981093855503}
              precision    recall  f1-score   support

          C0     0.5954    0.5712    0.5831       765
          C1     0.5936    0.6124    0.6029       725
          C2     0.6379    0.6415    0.6397       714
          C3     0.6049    0.6129    0.6089       762
          C4     0.5790    0.5760    0.5775       776
          C5     0.5731    0.5754    0.5743       749
          C6     0.5919    0.5832    0.5875       751
          C7     0.6039    0.5997    0.6018       722
          C8     0.6070    0.6096    0.6083       707
          C9     0.6119    0.6185    0.6152       734

    accuracy                         0.5996      7405
   macro avg     0.5999    0.6001    0.5999      7405
weighted avg     0.5995    0.5996  

In [27]:
# Section 9 - Diagnostics and plots
if history is not None:
    hist = pd.DataFrame(history.history)
else:
    hist_path = PATHS["metrics"] / "history.csv"
    hist = pd.read_csv(hist_path) if hist_path.exists() else pd.DataFrame()

if not hist.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist["loss"], label="train")
    axes[0].plot(hist["val_loss"], label="val")
    axes[0].set_title("Loss")
    axes[1].plot(hist["top1_acc"], label="train")
    axes[1].plot(hist["val_top1_acc"], label="val")
    axes[1].set_title("Top-1 accuracy")
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(alpha=0.35)
    plt.tight_layout()
    plt.savefig(PATHS["metrics"] / "training_curves.png", dpi=150, bbox_inches="tight")
    plt.close()

cm = confusion_matrix(data["test"]["y"].ravel(), test_probs.argmax(axis=-1).ravel(), labels=ALL_LABELS)
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="mako", vmin=0, vmax=1, xticklabels=[f"C{i}" for i in ALL_LABELS], yticklabels=[f"C{i}" for i in ALL_LABELS])
plt.title("Ray-ID CrossFormer normalized confusion")
plt.xlabel("Predicted shuffled slot")
plt.ylabel("True shuffled slot")
plt.tight_layout()
plt.savefig(PATHS["metrics"] / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()

# Candidate-ID hit analysis: seen vs OOV physical cells.
test_inputs, _, _ = arrays["test"]
oov_slot = test_inputs["candidate_ids"] == OOV_INDEX
true_slots = data["test"]["y"]
true_oov = np.take_along_axis(oov_slot, true_slots, axis=1)
pred_slots = test_probs.argmax(axis=-1)
correct = pred_slots == true_slots
print("True target OOV rate:", float(true_oov.mean()))
print("Accuracy on seen targets:", float(correct[~true_oov].mean()) if (~true_oov).any() else None)
print("Accuracy on OOV targets:", float(correct[true_oov].mean()) if true_oov.any() else None)

True target OOV rate: 0.01688048615800135
Accuracy on seen targets: 0.5756868131868131
Accuracy on OOV targets: 0.592


In [28]:
# Section 10 - Persist artifacts and metadata
model.save(FINAL_PATH)
metadata = {
    "experiment": "Ray-ID CrossFormer",
    "created": datetime.datetime.now().isoformat(),
    "notebook": "notebooks/modeling/07_ray_id_crossformer.ipynb",
    "architecture": "Temporal RF encoder + cell ID embeddings + serving flag + masked cell self-attention + horizon cross-attention",
    "cache": str(PATHS["cache"]),
    "previous_finetuned_metrics": previous.get("finetuned_metrics", {}),
    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
    "hp": HP,
    "vocab_size": VOCAB_SIZE,
    "train_id_count": len(train_ids),
    "best_checkpoint": str(CKPT_PATH),
    "final_model": str(FINAL_PATH),
    "metrics_dir": str(PATHS["metrics"]),
}
meta_path = PATHS["metrics"] / "ray_id_crossformer_metadata.json"
json.dump(metadata, open(meta_path, "w"), indent=2)
print("Saved:")
print(" model:", FINAL_PATH)
print(" best:", CKPT_PATH)
print(" metadata:", meta_path)

Saved:
 model: /home/wassimmchichi/Downloads/Handover_projects/models/ray_id_crossformer_final.keras
 best: /home/wassimmchichi/Downloads/Handover_projects/models/best_ray_id_crossformer.keras
 metadata: /home/wassimmchichi/Downloads/Handover_projects/metrics/ray_id_crossformer/ray_id_crossformer_metadata.json


In [29]:
# Section 11 - Quick per-window explanation

def explain_window(idx: int, horizon: int = 0, top_k: int = 5):
    split = "test"
    inputs, _, _ = arrays[split]
    one = {k: v[idx:idx + 1] for k, v in inputs.items()}
    pred = model.predict(one, verbose=0)[0, horizon]
    order = np.argsort(pred)[::-1]
    candidate_ids = data[split]["candidate_ids"][idx]
    true_slot = int(data[split]["y"][idx, horizon])
    rows = []
    for rank, slot in enumerate(order[:top_k], start=1):
        rows.append({
            "rank": rank,
            "slot": int(slot),
            "physical_cell_id": int(candidate_ids[slot]),
            "probability": float(pred[slot]),
            "is_true": bool(slot == true_slot),
            "is_oov_id": bool(inputs["candidate_ids"][idx, slot] == OOV_INDEX),
            "is_serving": bool(inputs["serving_flag"][idx, slot] > 0),
        })
    return pd.DataFrame(rows)

explain_window(0, horizon=0)

,rank,slot,physical_cell_id,probability,is_true,is_oov_id,is_serving
0,1,1,1684,0.524022,False,False,False
1,2,7,1801,0.258426,False,False,False
2,3,8,1515,0.168891,False,False,False
3,4,9,1795,0.031457,True,False,False
4,5,5,1281,0.011814,False,False,False
